In [ ]:
# @title 📦 Setup — Run this cell first! { display-mode: "form" }
# --- Google Colab setup ---
import os
if os.path.exists('/content') and not os.path.exists('/content/stable_diffusion_carlos'):
    print('📥 Cloning repository...')
    os.system('git clone https://github.com/eth-bmai-fs26/stable_diffusion_carlos.git /content/stable_diffusion_carlos')
if os.path.exists('/content/stable_diffusion_carlos'):
    os.chdir('/content/stable_diffusion_carlos')
    print(f'📂 Working directory: {os.getcwd()}')

%matplotlib inline
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import math

plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['figure.facecolor'] = 'white'

# Okabe-Ito accessible color palette
COLORS = {
    'text': '#E69F00',
    'image': '#0072B2',
    'connection': '#56B4E9',
    'failure': '#D55E00',
    'crossmodal': '#CC79A7',
    'neutral': '#999999',
}

print("✅ Setup complete.")


In [ ]:
# @title 🔧 Helper Functions — Data & Visualization { display-mode: "form" }

# --- Shape Dataset Generation ---
SHAPES = ['circle', 'square', 'triangle']
COLORS_MAP = {
    'red': (230, 25, 25),
    'blue': (25, 25, 230),
    'green': (25, 204, 25),
}
SIZES = {'small': 5, 'large': 10}
POSITIONS = {
    'center': (16, 16),
    'offset': [(8, 8), (8, 24), (24, 8), (24, 24)],
}

def generate_shape(shape, color, size='large', position='center', img_size=32):
    """Generate a 32x32x3 image with a colored shape. Returns numpy array [0,1]."""
    img = Image.new('RGB', (img_size, img_size), (0, 0, 0))
    draw = ImageDraw.Draw(img)
    rgb = COLORS_MAP[color]
    radius = SIZES[size]
    if position == 'center':
        cx, cy = 16, 16
    else:
        rng = np.random.RandomState()
        cx, cy = POSITIONS['offset'][rng.randint(len(POSITIONS['offset']))]
    if shape == 'circle':
        draw.ellipse([cx-radius, cy-radius, cx+radius, cy+radius], fill=rgb)
    elif shape == 'square':
        draw.rectangle([cx-radius, cy-radius, cx+radius, cy+radius], fill=rgb)
    elif shape == 'triangle':
        pts = [(cx, cy-radius), (cx-radius, cy+radius), (cx+radius, cy+radius)]
        draw.polygon(pts, fill=rgb)
    return np.array(img, dtype=np.float32) / 255.0

def generate_dataset(include_size=False, include_position=False):
    """Generate shape dataset. Base: 9 classes (3 shapes x 3 colors)."""
    sizes = list(SIZES.keys()) if include_size else ['large']
    positions = ['center', 'offset'] if include_position else ['center']
    dataset = []
    for color in COLORS_MAP:
        for shape in SHAPES:
            for sz in sizes:
                for pos in positions:
                    img = generate_shape(shape, color, size=sz, position=pos)
                    label = f"{sz} {color} {shape}" if include_size else f"{color} {shape}"
                    dataset.append((img, label))
    return dataset

class ShapeDataset(torch.utils.data.Dataset):
    def __init__(self, include_size=False, include_position=False):
        raw = generate_dataset(include_size=include_size, include_position=include_position)
        self.images, self.labels, self.label_texts = [], [], []
        unique_labels = []
        for img, label in raw:
            if label not in unique_labels:
                unique_labels.append(label)
        self.label_to_idx = {l: i for i, l in enumerate(unique_labels)}
        for img, label in raw:
            self.images.append(torch.from_numpy(img).permute(2, 0, 1))
            self.labels.append(self.label_to_idx[label])
            self.label_texts.append(label)
    def __len__(self):
        return len(self.images)
    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx], self.label_texts[idx]

# --- Visualization Helpers ---
def plot_image_grid(images, titles=None, ncols=3, figsize=None, suptitle=None):
    """Display a grid of images."""
    n = len(images)
    nrows = (n + ncols - 1) // ncols
    if figsize is None:
        figsize = (ncols * 2.5, nrows * 2.5)
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = np.array(axes).flatten() if isinstance(axes, np.ndarray) else [axes]
    for i, ax in enumerate(axes):
        if i < n:
            img = images[i]
            if isinstance(img, torch.Tensor):
                img = img.permute(1, 2, 0).detach().cpu().numpy() if img.dim() == 3 and img.shape[0] in (1,3) else img.detach().cpu().numpy()
            ax.imshow(np.clip(img, 0, 1))
            if titles and i < len(titles):
                ax.set_title(titles[i], fontsize=12)
        ax.axis('off')
    if suptitle:
        fig.suptitle(suptitle, fontsize=14, fontweight='bold')
    fig.tight_layout()
    return fig

def plot_similarity_matrix(sim_matrix, row_labels, col_labels, title='Cosine Similarity'):
    """Plot a heatmap with numeric annotations. Sequential blue colormap."""
    if isinstance(sim_matrix, torch.Tensor):
        sim_matrix = sim_matrix.detach().cpu().numpy()
    n_rows, n_cols = sim_matrix.shape
    fig, ax = plt.subplots(figsize=(max(6, n_cols*0.8), max(5, n_rows*0.8)))
    im = ax.imshow(sim_matrix, cmap='Blues', vmin=-1, vmax=1, aspect='auto')
    for i in range(n_rows):
        for j in range(n_cols):
            val = sim_matrix[i, j]
            color = 'white' if abs(val) > 0.5 else 'black'
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=10, color=color)
    ax.set_xticks(range(n_cols))
    ax.set_xticklabels(col_labels, rotation=45, ha='right', fontsize=10)
    ax.set_yticks(range(n_rows))
    ax.set_yticklabels(row_labels, fontsize=10)
    ax.set_title(title, fontsize=14, fontweight='bold')
    fig.colorbar(im, ax=ax, shrink=0.8)
    fig.tight_layout()
    return fig

def plot_denoising_filmstrip(images, steps=None, title='Denoising Process'):
    """Plot a horizontal filmstrip of denoising frames."""
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(n * 2, 2.5))
    if n == 1: axes = [axes]
    for i, (ax, img) in enumerate(zip(axes, images)):
        if isinstance(img, torch.Tensor):
            img = img.permute(1, 2, 0).detach().cpu().numpy() if img.dim() == 3 and img.shape[0] in (1,3) else img.detach().cpu().numpy()
        ax.imshow(np.clip(img, 0, 1))
        label = f't={steps[i]}' if steps else f'Step {i}'
        ax.set_title(label, fontsize=10)
        ax.axis('off')
    fig.suptitle(title, fontsize=14, fontweight='bold')
    fig.tight_layout()
    return fig

def plot_training_loss(losses, title='Training Loss'):
    """Plot training loss curve."""
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(losses, color=COLORS['image'], linewidth=2)
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Loss', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    return fig

def plot_embedding_space_2d(embeddings, labels, title='Embedding Space', colors_list=None):
    """Plot 2D PCA projection of embeddings."""
    if isinstance(embeddings, torch.Tensor):
        embeddings = embeddings.detach().cpu().numpy()
    centered = embeddings - embeddings.mean(axis=0)
    U, S, Vt = np.linalg.svd(centered, full_matrices=False)
    proj = centered @ Vt[:2].T
    fig, ax = plt.subplots(figsize=(8, 6))
    unique_labels = list(dict.fromkeys(labels))
    palette = ['#E69F00', '#0072B2', '#56B4E9', '#D55E00', '#CC79A7', '#009E73', '#F0E442', '#999999', '#000000']
    markers = ['o', 's', '^', 'D', 'v', 'P', '*', 'X', 'h']
    for i, ul in enumerate(unique_labels):
        idx = [j for j, l in enumerate(labels) if l == ul]
        ax.scatter(proj[idx, 0], proj[idx, 1], c=palette[i % len(palette)],
                   marker=markers[i % len(markers)], s=80, label=ul, edgecolors='black', linewidth=0.5)
    ax.set_xlabel('PC 1', fontsize=12)
    ax.set_ylabel('PC 2', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend(fontsize=9, loc='best', ncol=2)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    return fig

# --- Models needed for Cold Open demo ---
VOCAB = ['red','blue','green','circle','square','triangle',
         'small','large','center','offset','<pad>','<unk>']
WORD_TO_IDX = {w: i for i, w in enumerate(VOCAB)}

def tokenize(text, max_len=4):
    tokens = [WORD_TO_IDX.get(w, WORD_TO_IDX['<unk>']) for w in text.lower().split()]
    if len(tokens) < max_len:
        tokens += [WORD_TO_IDX['<pad>']] * (max_len - len(tokens))
    return torch.tensor(tokens[:max_len], dtype=torch.long)

class TextEncoder(nn.Module):
    def __init__(self, vocab_size=12, embed_dim=32, hidden_dim=64, out_dim=32):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=WORD_TO_IDX['<pad>'])
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, out_dim)
    def forward(self, token_ids):
        x = self.embedding(token_ids)
        mask = (token_ids != WORD_TO_IDX['<pad>']).unsqueeze(-1).float()
        x = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.normalize(x, dim=-1)

class ImageEncoder(nn.Module):
    def __init__(self, input_dim=3072, out_dim=32):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.fc2 = nn.Linear(64, out_dim)
    def forward(self, images):
        x = images.flatten(1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.normalize(x, dim=-1)

class CrossAttention(nn.Module):
    def __init__(self, img_dim=128, text_dim=32, head_dim=32):
        super().__init__()
        self.W_q = nn.Linear(img_dim, head_dim)
        self.W_k = nn.Linear(text_dim, head_dim)
        self.W_v = nn.Linear(text_dim, head_dim)
        self.W_out = nn.Linear(head_dim, img_dim)
        self.scale = head_dim ** -0.5
    def forward(self, image_features, text_features):
        Q = self.W_q(image_features)
        K = self.W_k(text_features)
        V = self.W_v(text_features)
        attn = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        attn_weights = F.softmax(attn, dim=-1)
        out = torch.matmul(attn_weights, V)
        out = self.W_out(out)
        return out, attn_weights

class MicroUNet(nn.Module):
    def __init__(self, time_emb_dim=32, text_emb_dim=32):
        super().__init__()
        self.enc1 = nn.Conv2d(3, 32, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, 32)
        self.enc2 = nn.Conv2d(32, 64, 3, stride=2, padding=1)
        self.norm2 = nn.GroupNorm(8, 64)
        self.enc3 = nn.Conv2d(64, 128, 3, stride=2, padding=1)
        self.norm3 = nn.GroupNorm(8, 128)
        self.time_mlp = nn.Sequential(nn.Linear(time_emb_dim, 128), nn.ReLU())
        self.cross_attn = CrossAttention(img_dim=128, text_dim=text_emb_dim, head_dim=32)
        self.dec1 = nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1)
        self.norm4 = nn.GroupNorm(8, 64)
        self.dec2 = nn.ConvTranspose2d(128, 32, 4, stride=2, padding=1)
        self.norm5 = nn.GroupNorm(8, 32)
        self.final = nn.Conv2d(64, 3, 3, padding=1)
    def forward(self, x, t_emb, text_emb):
        e1 = F.relu(self.norm1(self.enc1(x)))
        e2 = F.relu(self.norm2(self.enc2(e1)))
        e3 = F.relu(self.norm3(self.enc3(e2)))
        t = self.time_mlp(t_emb)
        e3 = e3 + t[:, :, None, None]
        B, C, H, W = e3.shape
        patches = e3.permute(0, 2, 3, 1).reshape(B, H*W, C)
        text_feat = text_emb.unsqueeze(1)
        attn_out, _ = self.cross_attn(patches, text_feat)
        e3 = attn_out.reshape(B, H, W, C).permute(0, 3, 1, 2)
        d1 = F.relu(self.norm4(self.dec1(e3)))
        d1 = torch.cat([d1, e2], dim=1)
        d2 = F.relu(self.norm5(self.dec2(d1)))
        d2 = torch.cat([d2, e1], dim=1)
        return self.final(d2)

class NoiseScheduler:
    def __init__(self, T=50, s=0.008):
        self.T = T
        steps = torch.arange(T + 1, dtype=torch.float64)
        f = torch.cos((steps / T + s) / (1 + s) * math.pi / 2) ** 2
        alpha_bars = (f / f[0]).clamp(min=1e-5, max=1.0)
        betas = (1 - alpha_bars[1:] / alpha_bars[:-1]).clamp(min=0.0001, max=0.999)
        self.betas = betas.float()
        self.alphas = (1 - self.betas).float()
        self.alpha_bars = torch.cumprod(self.alphas, dim=0).float()
    def add_noise(self, x_0, t, noise=None):
        if noise is None: noise = torch.randn_like(x_0)
        ab = self.alpha_bars[t]
        while ab.dim() < x_0.dim(): ab = ab.unsqueeze(-1)
        return torch.sqrt(ab) * x_0 + torch.sqrt(1 - ab) * noise
    def sample_step(self, x_t, t, predicted_noise):
        beta_t = self.betas[t]; alpha_t = self.alphas[t]; ab = self.alpha_bars[t]
        while beta_t.dim() < x_t.dim():
            beta_t = beta_t.unsqueeze(-1); alpha_t = alpha_t.unsqueeze(-1); ab = ab.unsqueeze(-1)
        mean = (1/torch.sqrt(alpha_t)) * (x_t - (beta_t/torch.sqrt(1-ab)) * predicted_noise)
        if (isinstance(t, int) and t == 0) or (isinstance(t, torch.Tensor) and (t==0).all()):
            return mean
        return mean + torch.sqrt(beta_t) * torch.randn_like(x_t)
    @staticmethod
    def get_time_embedding(t, dim=32):
        if isinstance(t, int): t = torch.tensor([t], dtype=torch.float32)
        elif isinstance(t, torch.Tensor) and t.dim() == 0: t = t.unsqueeze(0).float()
        else: t = t.float()
        half = dim // 2
        emb = torch.exp(torch.arange(half, dtype=torch.float32) * -(math.log(10000) / (half - 1)))
        emb = t.unsqueeze(1) * emb.unsqueeze(0)
        return torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)

@torch.no_grad()
def generate_image(model, scheduler, text_emb, null_text_emb=None,
                   guidance_scale=7.5, steps=50, seed=42):
    """Generate an image with optional classifier-free guidance."""
    torch.manual_seed(seed)
    x = torch.randn(1, 3, 32, 32)
    intermediates = []
    save_steps = set([steps-1] + [int(i*(steps-1)/7) for i in range(8)])
    for t_idx in reversed(range(steps)):
        t_emb_vec = scheduler.get_time_embedding(t_idx)
        if null_text_emb is not None and guidance_scale != 1.0:
            noise_cond = model(x, t_emb_vec, text_emb)
            noise_uncond = model(x, t_emb_vec, null_text_emb)
            predicted_noise = noise_uncond + guidance_scale * (noise_cond - noise_uncond)
        else:
            predicted_noise = model(x, t_emb_vec, text_emb)
        x = scheduler.sample_step(x, t_idx, predicted_noise)
        if t_idx in save_steps:
            intermediates.append(x[0].clone().clamp(0, 1))
    intermediates.reverse()
    return x[0].clamp(0, 1), intermediates

print("✅ Helper functions loaded.")


In [ ]:
# Runtime check — make sure setup cells have been run
try:
    _ = len(COLORS)
    _ = generate_shape
    print("✅ Ready to go!")
except NameError:
    print("⚠️ Please run the setup cells above (grey cells 1-2).")
    print("   Click the ▶ button on each grey cell at the top.")


# 📋 Quick Knowledge Check (Pre-Test)

Before we begin, let's see what you already know. **No right or wrong here** — this just helps you see how much you learn by the end.

Answer the 5 questions below, then scroll down to start the course.


In [ ]:
# Pre-test quiz — answer before starting the course
from ipywidgets import interact, RadioButtons, VBox, HTML
import ipywidgets as widgets

PRE_TEST_ANSWERS = {}

quiz_questions = [
    ("Q1", "What is the primary role of CLIP in a text-to-image system?",
     ["a) It generates the final image pixels directly from text",
      "b) It translates text and images into a shared language so they can be compared",
      "c) It removes noise from images during the generation process",
      "d) It compresses images to reduce file size"]),
    ("Q2", "Why does the same text prompt produce different images each time?",
     ["a) The model has bugs that cause random behavior",
      "b) The process starts from random noise, and different starting noise leads to different valid outputs",
      "c) The text is interpreted differently each time",
      "d) The model forgets previous outputs"]),
    ("Q3", "What mechanism allows the text prompt to influence which parts of the image get modified?",
     ["a) The text is converted to pixels and overlaid on the image",
      "b) A random selection process picks image regions to modify",
      "c) Cross-attention lets each image region check the text to decide what to generate",
      "d) The entire image is regenerated from scratch at each step"]),
    ("Q4", "A generated image shows the right colors but wrong shapes. What is the most likely cause?",
     ["a) The text encoder is broken",
      "b) The denoiser lacks spatial awareness (no convolutional layers)",
      "c) The noise schedule is too aggressive",
      "d) The image resolution is too low"]),
    ("Q5", "Which factor has the biggest impact on the cost of generating an image?",
     ["a) The length of the text prompt",
      "b) The number of denoising steps (each requires a full model pass)",
      "c) The color palette of the output image",
      "d) The file format of the saved image"]),
]

quiz_widgets = []
for qid, question, options in quiz_questions:
    label = widgets.HTML(value=f"<b>{qid}. {question}</b>")
    radio = RadioButtons(options=options, layout=widgets.Layout(width='auto'))
    def make_handler(q_id, r):
        def handler(change):
            PRE_TEST_ANSWERS[q_id] = change['new'][0]  # Store letter
        return handler
    radio.observe(make_handler(qid, radio), names='value')
    PRE_TEST_ANSWERS[qid] = options[0][0]  # Default to first option letter
    quiz_widgets.extend([label, radio, widgets.HTML(value="<br>")])

display(VBox(quiz_widgets))
print("Select your answers above, then continue to the course below.")


---

# 🎬 Notebook 1: The Raw Ingredients

## Images and Text as Numbers

*You're about to build a text-to-image AI model. But first, let's see one in action.*


In [ ]:
# Cold Open: See the model generate an image from text
import ipywidgets as widgets
from ipywidgets import interact

# Load pre-trained weights
_demo_ready = False
try:
    weights = torch.load('weights/full_pipeline.pt', map_location='cpu', weights_only=False)
    demo_text_enc = TextEncoder()
    demo_unet = MicroUNet()
    demo_text_enc.load_state_dict(weights['clip_text'])
    demo_unet.load_state_dict(weights['unet'])
    demo_scheduler = NoiseScheduler()
    demo_text_enc.eval(); demo_unet.eval()
    _demo_ready = True
    print("✅ Pre-trained model loaded.")
except Exception as e:
    print(f"⚠️ Could not load weights: {e}")
    print("   Please ensure weights/full_pipeline.pt is available.")
    print("   The interactive demo below will be skipped.")

if _demo_ready:
    prompts_list = ['red circle', 'red square', 'red triangle',
                    'blue circle', 'blue square', 'blue triangle',
                    'green circle', 'green square', 'green triangle']

    # Pre-compute null embedding for CFG
    null_tokens = tokenize('<pad> <pad>')
    null_emb = demo_text_enc(null_tokens.unsqueeze(0))

    @interact(prompt=widgets.Dropdown(options=prompts_list, value='red circle',
                                       description='Prompt:'),
              seed=widgets.IntSlider(min=0, max=999, value=42, description='Seed:'))
    def generate_demo(prompt, seed):
        text_emb = demo_text_enc(tokenize(prompt).unsqueeze(0))
        final, frames = generate_image(demo_unet, demo_scheduler, text_emb,
                                        null_emb, guidance_scale=7.5, seed=seed)
        step_labels = list(reversed([int(i*49/7) for i in range(8)]))
        plot_denoising_filmstrip(frames, steps=step_labels,
                                  title=f'Generating: "{prompt}" (seed={seed})')
        plt.show()


**You just ran a text-to-image model.** It has about 300,000 parameters — that's **12,000 times smaller** than Stable Diffusion XL, but it uses the **exact same architecture**.

By the end of this course, you'll understand every piece of how it works. Let's start with the raw ingredients: images and text.


---

# Part 1: What Computers See

*To a computer, every image is just a grid of numbers.*


In [ ]:
# A red circle: what YOU see vs what the COMPUTER sees
img = generate_shape('circle', 'red')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Left: the image
ax1.imshow(img)
ax1.set_title("What you see", fontsize=14, fontweight='bold')
ax1.axis('off')

# Right: the numbers (show red channel of a zoomed region)
region = img[10:22, 10:22, 0]  # Red channel, center region
im = ax2.imshow(region, cmap='Reds', vmin=0, vmax=1)
for i in range(region.shape[0]):
    for j in range(region.shape[1]):
        ax2.text(j, i, f'{region[i,j]:.1f}', ha='center', va='center',
                 fontsize=8, color='black' if region[i,j] < 0.5 else 'white')
ax2.set_title("What the computer sees (red channel)", fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()


**An image is a spreadsheet.** Each cell holds 3 numbers — one for Red, one for Green, one for Blue. A 32x32 image = 3,072 numbers. That's all there is to it.

> Think of it like this: every digital photo is just a very large spreadsheet with color values.


In [ ]:
# YOUR TURN: Drag the sliders to see how changing numbers changes the image
from ipywidgets import interact, FloatSlider

base_img = generate_shape('circle', 'red')

@interact(brightness=FloatSlider(min=0.0, max=2.0, step=0.1, value=1.0,
                                  description='Brightness:'),
          contrast=FloatSlider(min=0.5, max=1.5, step=0.1, value=1.0,
                                description='Contrast:'))
def adjust_image(brightness, contrast):
    adjusted = contrast * (base_img - 0.5) + 0.5 + (brightness - 1.0)
    adjusted = np.clip(adjusted, 0, 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    ax1.imshow(adjusted)
    ax1.set_title(f"Brightness={brightness:.1f}, Contrast={contrast:.1f}", fontsize=12)
    ax1.axis('off')
    region = adjusted[10:22, 10:22, 0]
    ax2.imshow(region, cmap='Reds', vmin=0, vmax=1)
    for i in range(region.shape[0]):
        for j in range(region.shape[1]):
            ax2.text(j, i, f'{region[i,j]:.1f}', ha='center', va='center',
                     fontsize=8, color='black' if region[i,j] < 0.5 else 'white')
    ax2.set_title("Red channel values", fontsize=12)
    fig.tight_layout()
    plt.show()


In [ ]:
# YOUR TURN: Change the shape and color, then run this cell
# Try: shape = 'square', color = 'blue'

shape = 'circle'    # Options: 'circle', 'square', 'triangle'
color = 'red'       # Options: 'red', 'blue', 'green'

img = generate_shape(shape, color)

plt.figure(figsize=(3, 3))
plt.imshow(img)
plt.title(f"{color} {shape}", fontsize=14)
plt.axis('off')
plt.show()


In [ ]:
# Our complete toy dataset: every combination of shape and color
dataset = generate_dataset()
images = [item[0] for item in dataset]
labels = [item[1] for item in dataset]

fig = plot_image_grid(images, titles=labels, ncols=3,
                       suptitle="Our Toy Dataset: 9 Classes")
plt.show()
print(f"Dataset size: {len(dataset)} images, each {images[0].shape}")
print("This is our toy dataset. Small enough to train on a laptop,")
print("big enough to learn the concepts behind Stable Diffusion.")


<div style="background-color: #FFF3CD; border-left: 4px solid #E69F00; padding: 15px; margin: 10px 0; border-radius: 4px;">

**💼 Manager's Briefing: Resolution = Cost**

Our toy images are 32x32 pixels = 3,072 numbers. Stable Diffusion generates 512x512 = 786,432 numbers. SDXL goes to 1024x1024 = 3,145,728 numbers.

**Key question to ask your engineering team:** *"What resolution does your model generate natively?"* Higher resolution = more compute = higher cost per image.

</div>


---

# Part 2: Words as Numbers

*Images are grids of numbers. But what about text? How do you turn "red circle" into numbers a computer can work with?*


In [ ]:
# What if words had addresses on a map?
# Here are 9 words placed in a 2D space based on meaning

word_embeddings = {
    'red':      np.array([0.9, 0.1, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0]),
    'blue':     np.array([0.1, 0.1, 0.9, 0.0, 0.0, 0.0, 0.0, 0.0]),
    'green':    np.array([0.1, 0.8, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0]),
    'circle':   np.array([0.0, 0.0, 0.0, 0.9, 0.1, 0.1, 0.0, 0.0]),
    'square':   np.array([0.0, 0.0, 0.0, 0.1, 0.9, 0.1, 0.0, 0.0]),
    'triangle': np.array([0.0, 0.0, 0.0, 0.1, 0.1, 0.9, 0.0, 0.0]),
    'small':    np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.2, 0.8]),
    'large':    np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.8, 0.2]),
    'round':    np.array([0.0, 0.0, 0.0, 0.8, 0.2, 0.1, 0.0, 0.0]),
}

all_embs = np.stack(list(word_embeddings.values()))
all_words = list(word_embeddings.keys())

fig = plot_embedding_space_2d(all_embs, all_words, title='Words as Points in Space')
plt.show()
print("Notice: color words cluster together, shape words cluster together.")
print("Words with similar meanings have similar 'addresses.'")


In [ ]:
# Cosine similarity: measuring how similar two word-addresses are
def cosine_similarity(a, b):
    """How similar are two vectors? Returns -1 to 1."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Let's check some pairs
pairs = [('red', 'blue'), ('red', 'circle'), ('circle', 'round')]
for w1, w2 in pairs:
    sim = cosine_similarity(word_embeddings[w1], word_embeddings[w2])
    print(f"  '{w1}' vs '{w2}': similarity = {sim:.3f}")

print("\n'red' and 'blue' are somewhat similar (both colors).")
print("'red' and 'circle' are very different (color vs shape).")
print("'circle' and 'round' are very similar (related meanings).")


In [ ]:
# YOUR TURN: Pick any two words and see how similar they are
from ipywidgets import interact, Dropdown

word_list = list(word_embeddings.keys())

@interact(word1=Dropdown(options=word_list, value='red', description='Word 1:'),
          word2=Dropdown(options=word_list, value='circle', description='Word 2:'))
def compare_words(word1, word2):
    sim = cosine_similarity(word_embeddings[word1], word_embeddings[word2])
    # Plot all words in 2D, highlight the selected pair
    all_embs = np.stack(list(word_embeddings.values()))
    centered = all_embs - all_embs.mean(axis=0)
    U, S, Vt = np.linalg.svd(centered, full_matrices=False)
    proj = centered @ Vt[:2].T

    fig, ax = plt.subplots(figsize=(8, 6))
    for i, w in enumerate(word_list):
        color = COLORS['neutral']
        size = 60
        if w == word1 or w == word2:
            color = COLORS['text'] if w == word1 else COLORS['image']
            size = 120
        ax.scatter(proj[i, 0], proj[i, 1], c=color, s=size, zorder=5,
                   edgecolors='black', linewidth=1)
        ax.annotate(w, (proj[i, 0], proj[i, 1]), fontsize=11,
                    ha='center', va='bottom', textcoords="offset points",
                    xytext=(0, 8))

    # Draw line between selected pair
    i1, i2 = word_list.index(word1), word_list.index(word2)
    ax.plot([proj[i1,0], proj[i2,0]], [proj[i1,1], proj[i2,1]],
            color=COLORS['connection'], linewidth=2, linestyle='--', zorder=4)

    ax.set_title(f'Cosine Similarity: {sim:.3f}', fontsize=14, fontweight='bold')
    ax.set_xlabel('PC 1', fontsize=12); ax.set_ylabel('PC 2', fontsize=12)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()


In [ ]:
# STUMBLE: Can we compare text and images directly?
# Let's try projecting images into the same space as text embeddings

np.random.seed(42)
random_proj = np.random.randn(3072, 8) / np.sqrt(3072)  # Random projection

# Project images into 8D space (same as our word embeddings)
dataset = generate_dataset()
img_projected = []
for img, label in dataset:
    flat = img.flatten()
    img_projected.append(flat @ random_proj)
img_projected = np.stack(img_projected)

# Compute similarity between text embeddings and projected images
text_labels = [item[1] for item in dataset]
text_embs = np.stack([
    np.mean([word_embeddings[w] for w in label.split()], axis=0)
    for label in text_labels
])

# Cosine similarity matrix
def cos_sim_matrix(A, B):
    A_norm = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-8)
    B_norm = B / (np.linalg.norm(B, axis=1, keepdims=True) + 1e-8)
    return A_norm @ B_norm.T

sim = cos_sim_matrix(text_embs, img_projected)

fig = plot_similarity_matrix(sim, text_labels, text_labels,
                              title='Text vs Image Similarity (Random Projection)')
# Add vermillion border for stumble
for spine in fig.axes[0].spines.values():
    spine.set_edgecolor(COLORS['failure'])
    spine.set_linewidth(3)
plt.show()

print("❌ The similarity matrix looks random — no clear diagonal pattern.")
print("   Text embeddings and image pixels live in completely different spaces.")
print("   Random projection can\'t bridge that gap.")


---

### 💡 The Key Insight

**Text-space and image-space don't speak the same language.** Random projection can't fix that — the spaces are fundamentally different.

> **Co-pilot metaphor:** Imagine a driver (the image model) and a co-pilot (the text). Right now, the driver speaks French and the co-pilot speaks Japanese. They can't communicate. **CLIP teaches them a common language.**

In the next notebook, we'll build CLIP — the system that learns to place matching text and images at the same "address" in a shared space.


<div style="background-color: #FFF3CD; border-left: 4px solid #E69F00; padding: 15px; margin: 10px 0; border-radius: 4px;">

**💼 Manager's Briefing: Joint vs Post-Hoc Alignment**

There are two ways to connect text and images:

1. **Post-hoc** (what we just tried): Build separate text and image systems, then try to connect them later. This rarely works well.
2. **Joint training** (CLIP): Train text and image encoders together from the start, so they learn a shared representation.

**Key question for vendors:** *"Was your text encoder trained jointly with your image model, or bolted on afterward?"* Joint training produces much better text-image alignment.

</div>


---

## 📋 Co-Pilot Reference Card

| Notebook | Component | What It Does | Co-Pilot Metaphor |
|----------|-----------|-------------|-------------------|
| **NB1** | **Raw Ingredients** | **Images = tensors of numbers; text = embedding vectors; cosine sim measures closeness** | **"Here are the parts of the car and the map"** |

*This card grows with each notebook. By Notebook 5, you'll have a complete decision-maker's cheat sheet.*